# Step 2: Exploratory Data Analysis & Quality Inspection
**Project:** SC07 Smart Irrigation System  
**Objective:** To systematically load, inspect, and validate the raw agricultural dataset prior to formal preprocessing. Ensuring data integrity at this stage prevents downstream propagation of anomalies into the predictive pipeline.

## 1. Environment Setup & Raw Data Ingestion
We establish a robust path configuration to locate the raw dataset dynamically. Defensive checks are incorporated to verify file existence, followed by a primary diagnostic report covering data dimensions, memory footprint, and duplication metrics.

In [4]:
import sys
from pathlib import Path
import pandas as pd

print("[INFO] Initializing environment and configuring file paths...")
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "irrigation_prediction.csv"


if not RAW_FILE.exists():
    print(f"[ERROR] Raw data file not found at: {RAW_FILE}")
    sys.exit("[CRITICAL] Cannot proceed without raw data.")

print(f"[SUCCESS] Located raw data at: {RAW_FILE.name}")
print("[INFO] Loading dataset into memory...")

try:
    df = pd.read_csv(RAW_FILE)
    

    print("\n" + "=" * 55)
    print("SYSTEM LOG: RAW DATASET INSPECTION")
    print("=" * 55)
    print(f" -> Dataset Dimensions: {df.shape[0]:,} records, {df.shape[1]} features")
    

    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f" -> Memory Footprint:   {memory_mb:.2f} MB")
    
    print(f" -> Duplicate Records:  {df.duplicated().sum()}")
    print("-" * 55)
    
    print("[INFO] Conducting null-value audit across all features...")
    missing_data = df.isnull().sum()
    if missing_data.sum() == 0:
        print(" -> [OK] No missing values detected in the dataset.")
    else:
        print(" -> [WARNING] Missing values detected:\n", missing_data[missing_data > 0])
        
    print("-" * 55)
    print("[INFO] Displaying initial dataset preview (Top 5 rows):")
    display(df.head())
    
except Exception as e:
    print(f"[ERROR] Failed to read or process the dataset: {str(e)}")

[INFO] Initializing environment and configuring file paths...
[SUCCESS] Located raw data at: irrigation_prediction.csv
[INFO] Loading dataset into memory...

SYSTEM LOG: RAW DATASET INSPECTION
 -> Dataset Dimensions: 10,000 records, 20 features
 -> Memory Footprint:   5.52 MB
 -> Duplicate Records:  0
-------------------------------------------------------
[INFO] Conducting null-value audit across all features...
 -> [OK] No missing values detected in the dataset.
-------------------------------------------------------
[INFO] Displaying initial dataset preview (Top 5 rows):


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


## 2. Data Cleansing & Imputation Strategy
Raw datasets often contain anomalies such as duplicate entries or missing values which can distort model training. In this phase, we scrub the dataset by removing exact duplicate rows to prevent bias. For any missing numerical data, we apply **Median Imputation** rather than mean imputation, as the median is highly robust against outliers in environmental data (e.g., sudden spikes in rainfall or temperature).

In [5]:
print("[INFO] Initiating data cleansing and imputation protocol...")
clean_df = df.copy()


initial_rows = len(clean_df)
duplicates_before = int(clean_df.duplicated().sum())
missing_before = clean_df.isna().sum()
total_missing_before = missing_before.sum()

print("[INFO] Purging duplicate records from the dataset...")
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("[INFO] Applying median imputation to numerical features...")
numeric_cols = clean_df.select_dtypes(include=['number']).columns

imputed_cols = missing_before[missing_before > 0].index.tolist()


clean_df[numeric_cols] = clean_df[numeric_cols].fillna(clean_df[numeric_cols].median())


assert clean_df.duplicated().sum() == 0, "[CRITICAL] Duplicate removal failed."
assert clean_df[numeric_cols].isna().sum().sum() == 0, "[CRITICAL] Imputation failed to resolve all NaNs."


print("\n" + "=" * 55)
print("DATA CLEANSING AUDIT REPORT")
print("=" * 55)
print(f" -> Initial Record Count:         {initial_rows:,}")
print(f" -> Final Record Count:           {len(clean_df):,}")
print(f" -> Duplicate Rows Removed:       {duplicates_before:,}")
print(f" -> Total Missing Values Fixed:   {total_missing_before:,}")
if imputed_cols:
    print(f" -> Columns Imputed (Median):     {', '.join(imputed_cols)}")
else:
    print(" -> Columns Imputed:              None required")
print("-" * 55)
print("[SUCCESS] Dataset is sanitized and validated. Ready for downstream analysis.")

[INFO] Initiating data cleansing and imputation protocol...
[INFO] Purging duplicate records from the dataset...
[INFO] Applying median imputation to numerical features...

DATA CLEANSING AUDIT REPORT
 -> Initial Record Count:         10,000
 -> Final Record Count:           10,000
 -> Duplicate Rows Removed:       0
 -> Total Missing Values Fixed:   0
 -> Columns Imputed:              None required
-------------------------------------------------------
[SUCCESS] Dataset is sanitized and validated. Ready for downstream analysis.


## 3. Dataset Stratification & Serialization
To rigorously evaluate our machine learning models, we must divide our sanitized dataset into three distinct subsets: **Training (70%)**, **Validation (15%)**, and **Testing (15%)**. 
We utilize **stratified sampling** based on our target variable (`Irrigation_Need`) to guarantee that the class distribution remains consistent across all splits. This prevents scenarios where a split might inadvertently contain an imbalance of classes. Finally, we serialize these splits into structured `.csv` files for downstream model consumption.

In [6]:
from sklearn.model_selection import train_test_split

print("[INFO] Preparing to partition dataset into Train, Validation, and Test splits...")


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
target_col = 'Irrigation_Need'

print(f"[INFO] Target variable identified as: '{target_col}'. Initiating stratified split...")

try:

    train_df, temp_df = train_test_split(
        clean_df, test_size=0.30, stratify=clean_df[target_col], random_state=42
    )
    
 
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df[target_col], random_state=42
    )

    print("[INFO] Serializing datasets to the processed directory...")

    clean_df.to_csv(PROCESSED_DIR / "irrigation_clean.csv", index=False)
    train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
    val_df.to_csv(PROCESSED_DIR / "validation.csv", index=False)
    test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)
    
 
    train_dist = train_df[target_col].value_counts(normalize=True).round(4) * 100
    

    print("\n" + "=" * 55)
    print("DATASET PARTITION & EXPORT SUMMARY")
    print("=" * 55)
    print(f" -> Master Cleaned Dataset:  {len(clean_df):,} records")
    print("-" * 55)
    print(f" -> Training Split (70%):    {len(train_df):,} records")
    print(f" -> Validation Split (15%):  {len(val_df):,} records")
    print(f" -> Testing Split (15%):     {len(test_df):,} records")
    print("-" * 55)
    print(f" -> Stratification Target:   '{target_col}'")
    print(f" -> Train Class Imbalance:   {train_dist.to_dict()} (%)")
    print(f" -> Fixed Random Seed:       42")
    try:

        print(f" -> Export Path:             {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")
    except ValueError:
        print(f" -> Export Path:             {PROCESSED_DIR}")
    print("=" * 55)
    print("[SUCCESS] Data pipeline execution completed. Assets serialized successfully.")

except Exception as e:
    print(f"[ERROR] An issue occurred during data splitting or saving: {str(e)}")

[INFO] Preparing to partition dataset into Train, Validation, and Test splits...
[INFO] Target variable identified as: 'Irrigation_Need'. Initiating stratified split...
[INFO] Serializing datasets to the processed directory...

DATASET PARTITION & EXPORT SUMMARY
 -> Master Cleaned Dataset:  10,000 records
-------------------------------------------------------
 -> Training Split (70%):    7,000 records
 -> Validation Split (15%):  1,500 records
 -> Testing Split (15%):     1,500 records
-------------------------------------------------------
 -> Stratification Target:   'Irrigation_Need'
 -> Train Class Imbalance:   {'Low': 58.64, 'Medium': 38.0, 'High': 3.36} (%)
 -> Fixed Random Seed:       42
 -> Export Path:             data/processed
[SUCCESS] Data pipeline execution completed. Assets serialized successfully.
